Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence

# Báo cáo Bài tập tuần - Buổi 6
**Nguyễn Đức Phát - 24110296**

---

## 1. Vấn đề bài toán (8-Puzzle Problem)
8-Puzzle là bài toán sắp xếp 8 ô số trên bảng 3x3 chứa các số từ 1 đến 8 và một ô trống (biểu diễn bằng số 0). Mục tiêu là di chuyển ô trống để đưa trạng thái bắt đầu (Start State) về trạng thái đích (Goal State) với số bước đi tối ưu.

## 2. Ý tưởng của thuật toán IDS (Iterative Deepening Search)
**Iterative Deepening Search (IDS)** hay Tìm kiếm sâu dần là một chiến lược kết hợp ưu điểm của cả **BFS** và **DFS**:
- **Ưu điểm của BFS**: Đảm bảo tìm thấy lời giải ngắn nhất (tối ưu) đầu tiên.
- **Ưu điểm của DFS**: Cực kỳ tiết kiệm bộ nhớ (độ phức tạp không gian chỉ là tuyến tính theo độ sâu của nhánh tìm kiếm).

### Cơ chế hoạt động:
1. Thuật toán thiết lập giới hạn độ sâu tối đa $L$ xuất phát từ $0$.
2. Thực hiện thuật toán **Tìm kiếm theo chiều sâu hạn chế (Depth-Limited Search - DLS)** với giới hạn độ sâu là $L$.
3. Nếu tìm thấy lời giải, thuật toán kết thúc và trả về đường đi tối ưu.
4. Nếu DLS duyệt hết tất cả các trạng thái trong giới hạn độ sâu $L$ mà không tìm thấy đích, ta tăng giới hạn độ sâu $L = L + 1$ và thực hiện lại DLS từ trạng thái bắt đầu.

### Trình tự ưu tiên duyệt LRUD:
Trình tự di chuyển của ô trống được ưu tiên duyệt theo thứ tự: **Left (Trái) $\rightarrow$ Right (Phải) $\rightarrow$ Up (Lên) $\rightarrow$ Down (Xuống)**.
- Trong cấu trúc Stack (ngăn xếp), để các trạng thái được lấy ra duyệt (`pop`) theo đúng thứ tự **LRUD**, các trạng thái lân cận sẽ được đẩy vào Stack (`push`) theo thứ tự đảo ngược là **Down $\rightarrow$ Up $\rightarrow$ Right $\rightarrow$ Left**.

## 3. Độ phức tạp thuật toán
- **Độ phức tạp thời gian**: $O(b^d)$, tương tự BFS, trong đó $b$ là hệ số nhánh (tối đa là 4) và $d$ là độ sâu lời giải ngắn nhất. Việc duyệt lại các nút ở các tầng trên trong mỗi lần lặp thực chất tốn rất ít chi phí so với sự bùng nổ số nút ở tầng cuối cùng.
- **Độ phức tạp không gian**: $O(b \cdot d)$, tuyến tính giống DFS, do chỉ cần lưu trữ nhánh tìm kiếm hiện tại trên Stack thay vì lưu toàn bộ các nút đã tạo như BFS.

## 4. Hướng dẫn chạy chương trình
1. Mở file `BTVN.ipynb` trong thư mục `Buổi 6` trên VS Code hoặc Jupyter Notebook.
2. Chạy cell code bên dưới để khởi chạy chương trình mô phỏng Tkinter.
3. Giao diện chương trình bao gồm:
   - **Current State**: Trạng thái hiện tại đang được xét duyệt.
   - **Frontier (Stack)**: Tập biên chứa các trạng thái đang chờ duyệt.
   - **Current Path (Ancestors)**: Đường đi từ nút gốc đến nút hiện tại (dùng để kiểm tra tránh tạo chu trình lặp).
   - **Bảng điều khiển**: Nút **Next Step** để duyệt thủ công từng bước, nút **Auto Run** (chuyển sang **Pause** để dừng lại) và thanh trượt **Speed** để điều chỉnh tốc độ duyệt thời gian thực (Real-time).

In [ ]:
# =========================================================================
# CHƯƠNG TRÌNH GIẢI BÀI TOÁN 8-PUZZLE BẰNG THUẬT TOÁN ITERATIVE DEEPENING SEARCH (IDS)
# - Kết hợp ưu điểm tối ưu của BFS và tiết kiệm không gian của DFS.
# - Trình tự ưu tiên duyệt lân cận: LRUD (Left, Right, Up, Down).
# - Trực quan hóa thời gian thực (Real-time) từng bước thông qua Tkinter.
# - Hỗ trợ thanh cuộn ngang, điều chỉnh tốc độ (Speed) và tự động chạy (Auto Run).
# =========================================================================

import tkinter as tk

START_STATE = ((2, 8, 3), (1, 6, 4), (7, 0, 5))
GOAL_STATE  = ((1, 2, 3), (8, 0, 4), (7, 6, 5))
MOVES = [
    (0, -1, 'Left'),
    (0, 1, 'Right'),
    (-1, 0, 'Up'),
    (1, 0, 'Down')
]

# ── helpers ──────────────────────────────────────────────
def get_neighbors(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    result = []
    for dx, dy, act in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            s = [list(r) for r in state]
            s[x][y], s[nx][ny] = s[nx][ny], s[x][y]
            result.append((tuple(tuple(r) for r in s), act))
    return result

def reconstruct(node):
    path = []
    while node:
        path.append(node['state'])
        node = node['parent']
    return path[::-1]

# ── DLS generator ────────────────────────────────────────
def dls_gen(start, goal, limit):
    # Stack chứa các dictionary: {'state': state, 'parent': parent, 'depth': depth}
    stack = [{'state': start, 'parent': None, 'depth': 0}]
    expanded = 0
    generated = 0
    while stack:
        node = stack.pop()
        state = node['state']
        depth = node['depth']
        
        # Kiểm tra tránh chu trình lặp trên đường đi hiện tại
        ancestors = []
        p = node['parent']
        while p:
            ancestors.append(p['state'])
            p = p['parent']
            
        if state in ancestors:
            continue
            
        expanded += 1
        frontier = [n['state'] for n in stack]
        
        yield state, frontier, ancestors, depth, limit, expanded, generated, None
        
        if state == goal:
            path = reconstruct(node)
            yield state, [], ancestors, depth, limit, expanded, generated, path
            return
            
        if depth < limit:
            neighbors = get_neighbors(state)
            # Do dùng Stack (LIFO), để duyệt theo thứ tự LRUD thì phải push vào theo thứ tự ngược lại (Down, Up, Right, Left)
            for nb, _ in reversed(neighbors):
                # Kiểm tra lặp sớm với các nút tổ tiên trên nhánh để giảm tải cho Stack
                p_check = node
                is_cycle = False
                while p_check:
                    if nb == p_check['state']:
                        is_cycle = True
                        break
                    p_check = p_check['parent']
                if not is_cycle:
                    stack.append({'state': nb, 'parent': node, 'depth': depth + 1})
                    generated += 1

# ── IDS generator ────────────────────────────────────────
def ids_gen(start, goal, max_limit=50):
    total_expanded = 0
    total_generated = 0
    for limit in range(max_limit + 1):
        dls = dls_gen(start, goal, limit)
        found = False
        for current, frontier, ancestors, depth, lim, exp, gen, path in dls:
            total_expanded += 1
            total_generated += gen
            yield current, frontier, ancestors, depth, limit, total_expanded, total_generated, path
            if path is not None:
                found = True
                break
        if found:
            return

# ── Scrollable Frame ─────────────────────────────────────
class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=180):
        super().__init__(parent)
        self.h_scroll = tk.Scrollbar(self, orient='horizontal')
        self.h_scroll.pack(side='bottom', fill='x')
        self.canvas = tk.Canvas(self, height=height)
        self.canvas.pack(side='left', fill='both', expand=True)
        self.h_scroll.config(command=self.canvas.xview)
        self.canvas.configure(xscrollcommand=self.h_scroll.set)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda e: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0, 0), window=self.inner, anchor='nw')
        self.canvas.bind('<Shift-MouseWheel>', lambda e: self.canvas.xview_scroll(int(-1 * (e.delta / 120)), 'units'))

# ── GUI ──────────────────────────────────────────────────
class PuzzleGUI:
    def __init__(self, root, gen):
        self.root = root
        self.gen  = gen
        self.running = False
        self.step_count = 0
        root.title('8 Puzzle IDS – Real-time Visualizer')
        
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw * 0.85)), min(900, int(sh * 0.85))
        root.geometry(f'{w}x{h}+{(sw - w) // 2}+{(sh - h) // 2}')
        root.minsize(900, 600)

        self.info = tk.Label(root, text='Press Auto Run or Next Step to begin IDS', font=('Arial', 14, 'bold'))
        self.info.pack(pady=6)

        self.cur_frame = tk.LabelFrame(root, text='Current State', padx=8, pady=8)
        self.cur_frame.pack(fill='x', padx=10, pady=4)

        fl = tk.LabelFrame(root, text='Frontier (Stack - Bottom to Top)', padx=8, pady=8)
        fl.pack(fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(fl, 180)
        self.frontier_sf.pack(fill='both', expand=True)

        el = tk.LabelFrame(root, text='Current Path (Ancestors - Root to Parent)', padx=8, pady=8)
        el.pack(fill='both', expand=True, padx=10, pady=4)
        self.explored_sf = ScrollableFrame(el, 180)
        self.explored_sf.pack(fill='both', expand=True)

        bf = tk.Frame(root)
        bf.pack(pady=8)
        tk.Button(bf, text='Next Step', width=14, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(bf, text='Auto Run', width=14, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=200)
        tk.Label(bf, text='Speed (ms):').pack(side='left', padx=(15, 2))
        tk.Scale(bf, from_=10, to=1000, orient='horizontal', variable=self.speed_var, length=160).pack(side='left')

    def draw_state(self, parent, state):
        f = tk.Frame(parent)
        for i in range(3):
            for j in range(3):
                v = state[i][j]
                tk.Label(f, text=str(v) if v else '', width=4, height=2,
                         font=('Arial', 13, 'bold'), relief='solid',
                         bg='#4CAF50' if v == 0 else 'white').grid(row=i, column=j)
        return f

    def clear(self, frame):
        for w in frame.winfo_children(): w.destroy()

    def render(self, current, frontier, ancestors, depth, limit, expanded, generated, path):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)
        self.clear(self.explored_sf.inner)

        self.draw_state(self.cur_frame, current).pack()
        for i, s in enumerate(frontier):
            self.draw_state(self.frontier_sf.inner, s).grid(row=0, column=i, padx=8, pady=8)
            
        # Hiển thị đường đi hiện tại (ancestors đảo ngược để từ gốc đến nút cha)
        for i, s in enumerate(reversed(ancestors)):
            self.draw_state(self.explored_sf.inner, s).grid(row=0, column=i, padx=8, pady=8)

        status = f'Step {self.step_count} | Limit (L): {limit} | Depth: {depth} | Expanded: {expanded} | Generated: {generated}'
        if path:
            status += f' | GOAL FOUND! Path length: {len(path)-1}'
        self.info.config(text=status)

    def next_step(self):
        try:
            current, frontier, ancestors, depth, limit, expanded, generated, path = next(self.gen)
            self.step_count += 1
            self.render(current, frontier, ancestors, depth, limit, expanded, generated, path)
            if path:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running: return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)

root = tk.Tk()
app = PuzzleGUI(root, ids_gen(START_STATE, GOAL_STATE))
root.mainloop()